In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,accuracy_score

this is a dataset on ai impact on jobs and layoff risk across industries .
Objective is to analyze the relationship between ai adoptions , task automation , workforce characterstics and employment risk.
building classifications model to predict layoff and risk 

In [ ]:
df=pd.read_csv("D:/yash - downloads/dataset/ai-impact-jobs-layoff-risk-dataset.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.info()
df.describe()


In [ ]:
df.isnull().sum()

In [ ]:
corr_matrix=df.corr(numeric_only=True) # this will work as it drops columns have non numeric data
print(corr_matrix)


In [ ]:
#ploting heatmap ( visulisatio of correlations matrix )
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('correlation matrix of features')
plt.show()


In [ ]:
#droping these columns as they likely may cause overfiting because of there high correlations 
features_to_drop=['Creativity_Requirement','Tasks_Automated_Percentage']
df=df.drop(columns=features_to_drop,errors='ignore')


In [ ]:
updated=df.drop(columns='Layoff_Risk')
#before spliting and asigning layoff column to y we need to change string int int 
target_mapping={'Low':0,'Medium':1,'High':2}
df['Layoff_Risk']=df['Layoff_Risk'].map(target_mapping)
y=df['Layoff_Risk']

In [ ]:
#encodeling the features have dtype string 
df_encoded=pd.get_dummies(updated,drop_first=True)
df_encoded.head()
print(df_encoded.columns.tolist())
X=df_encoded

In [ ]:

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
x_train=scaler.fit_transform(X_train)
x_test=scaler.fit_transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(x_train,y_train)

y_pred=model.predict(x_test)

In [ ]:

print("model accuracy : ",accuracy_score(y_test,y_pred))
print("\nDetailed classsification report: \n",classification_report(y_test,y_pred))

In [ ]:
cofficients=model.coef_[0]

feature_name=X.columns

importance_df=pd.DataFrame({'Feature':feature_name,'Weight':cofficients}).sort_values(by='Weight',ascending=False)
print(importance_df)

In [ ]:
plt.figure(figsize=(10,10))
colors=['crimson'if w>0 else 'royalblue' for w in importance_df['Weight']]
plt.barh(importance_df['Feature'],importance_df['Weight'],color=colors)
plt.axvline(0, color='black',linestyle='--')
plt.xlabel('Coefficient weight (Impact on layoff Risk)')
plt.title('feature Importance for layoff risk')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
feature_name=X.columns if 'X' in locals() else df_encoded.drop(columns=['Layoff_Risk'],errors='ignore').columns

avg_importance=np.mean(np.abs(model.coef_),axis=0)

importance_df=pd.DataFrame({'Feature':feature_name,'Importance':avg_importance}).sort_values(by='Importance',ascending=False)

plt.figure(figsize=(10,10))
sns.barplot(x='Importance',y='Feature',data=importance_df.head(10),palette='viridis')
plt.title("top 10 most important features(logistic regresssion)")
plt.xlabel('Average Absolute Coefficient weight')
plt.ylabel('feature')
plt.tight_layout()
plt.show()